### benötigte Bibliotheken herunterladen

pip install -r requirements.txt

## Aktuelles CSV einlesen

In [ ]:
""" import requests
import pandas as pd
import io
from pathlib import Path

OUTPUT_DIR = Path.cwd()

url = "https://data.bs.ch/api/explore/v2.1/catalog/datasets/100189/exports/csv"
params = {"delimiter": ";"}

df = None
try:
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), sep=";")
    print(df.shape)
    print(df.head())
except Exception:
    print("Server im Basel down") """

In [ ]:
""" import re

def _norm(s: str) -> str:
    #Vereinheitlicht Spaltennamen für den Vergleich.
    s = str(s).strip().lstrip("\ufeff").lower()
    s = s.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")
    return re.sub(r"[^a-z0-9]", "", s)

# Zielspalte -> akzeptierte Schreibweisen (der Zielname selbst zählt automatisch auch)
ALIASES = {
    "Id Strasse":             ["id_strasse", "id-strasse", "id strasse"],
    "Strassenname":           ["strassenname", "strasse"],
    "Erklärung erste Zeile":  ["erklaerung_erste_zeile"],
    "Erklärung zweite Zeile": ["erklaerung_zweite_zeile"],
    "Geo Shape":              ["geo_shape"],
    "Geo Point":              ["geo_point_2d", "geo_point"],
    "Erstmals erwähnt":       ["erstmalige_erwaehnung", "erstmals_erwaehnt"],
    "Amtlich benannt":        ["amtliche_benennung", "amtlich_benannt"],
    "Indextext":              ["strassenindex", "index"],
    "Kurztext":               ["strassenname_kurz", "kurz"],
    "Gemeindename":           ["Gemeindename", "gemeinde"],
}
ZIEL_SPALTEN = list(ALIASES.keys())

if df is not None:
    # Nachschlage-Tabelle: normalisierter Name -> Zielspalte
    lookup = {}
    for ziel, namen in ALIASES.items():
        for n in [ziel] + namen:
            lookup[_norm(n)] = ziel

    neue_namen, unbekannt = {}, []
    for col in df.columns:
        key = _norm(col)
        if key in lookup:
            neue_namen[col] = lookup[key]
        else:
            unbekannt.append(col)

    df = df.rename(columns=neue_namen)

    # --- Warnungen, damit du Änderungen von Basel sofort bemerkst ---
    fehlend = [z for z in ZIEL_SPALTEN if z not in set(neue_namen.values())]
    if fehlend:
        print("WARNUNG: Diese Zielspalten wurden NICHT gefunden und bleiben LEER:", fehlend)
    if unbekannt:
        print("HINWEIS: Diese Quellspalten ist unbekannt und wird leer gelassen:", unbekannt)
    elif unbekannt:
        print("Alle Kategorien wurden erkannt.")

    df = df.reindex(columns=ZIEL_SPALTEN) """

In [ ]:
""" # Auf Basel filtern – Spalte 'Gemeindename' robust finden
gem_col = next((c for c in df.columns if _norm(c) in ("gemeindename", "gemeinde")), None)
if gem_col is None:
    raise SystemExit("Spalte 'Gemeindename' nicht gefunden!")
df = df[df[gem_col].fillna("").str.strip().str.casefold() == "basel"].copy() """

In [ ]:
""" # CSV speichern – ersetzt vorhandene Datei automatisch
if df is not None:
    output_path = OUTPUT_DIR / "Strassen_export_01.csv"
    try:
        df.to_csv(output_path, sep=";", index=False, encoding="utf-8-sig")
        print(f"Gespeichert unter: {output_path}")
    except PermissionError:
        print("Datei noch geöffnet -> Datei noch schliessen erst.")
else:
    print("Nichts zu speichern.") """

# Fallback: CSV einlesen

In [ ]:
""" import pandas as pd

data = pd.read_csv("data/100189_raw.csv", delimiter=";") """

In [ ]:
""" # geopandas einlesen

import geopandas as gpd

data = gpd.read_file("data/100189_raw.csv") """

In [ ]:
import pandas as pd
import geopandas as gpd
import shapely

data = gpd.GeoDataFrame(
    pd.read_csv(
        "data/100189_raw.csv",
        sep=";",  
        converters={
            "Geo Shape": shapely.from_geojson
        }, 
        index_col=0,
    ),
    geometry="Geo Shape",  
    crs=4326,
)

data.plot()

In [ ]:
data.head()

## Bettingen und Riehen wegschneiden
mit Geopandas und Clip

In [ ]:
import geopandas as gpd

basel_grenzen = gpd.read_file("data/Basel_Boundarie.gpkg")
basel_grenzen.plot()


In [ ]:
data = gpd.clip(data, basel_grenzen)
data.plot()

## Beschreibung zusammensetzen

In [ ]:
for column in data.columns:
    if column.startswith("Erklärung"):
        data["Erklärung_komplett"] = data.get("Erklärung_komplett", "") + " " + data[column].fillna("")

In [ ]:
data.head()

## Klassifizieren

Männlich / Weiblich und Kategorien

In [ ]:
# Kategorien Keywords (Berufsgruppen, Gebäude, Epochen)
PROFESSIONS = {
    'Wissenschaft': [
        'professor', 'doktor', 'physiker', 'mathematiker', 'chemiker',
        'biologe', 'naturwissenschaft', 'gelehrte', 'forscher', 'wissenschaftler',
        'astronome', 'geologe', 'botaniker', 'zoolog', 'arzt', 'ärztin',
        'medizin', 'pharmazie', 'universität', 'akademie', 'ampere','chirurg','aufklärer',
        'ingenieur', 'erfinder','philosoph',
    ],
    'Kunst': [
        'maler', 'bildhauer', 'künstler', 'kunstgewerbe', 'kunsthandwerker',
        'kunstmaler', 'kunstbildner', 'kunstschule', 'kunstweg',
        'kunsthalle', 'galerie', 'kunstmuseum', 'kunstsammler',
        'dichter', 'schriftsteller', 'schriftstellerei', 'literatur',
        'komponist', 'musiker', 'musikant', 'sänger', 'gesang',
        'tänzer', 'schauspiel', 'theater', 'bühne', 'opernhaus', 'poet','kunst',
    ],
    'Handwerk': [
        'bauer', 'fischer', 'müller', 'schuster', 'schmied', 'zimmer',
        'zimmermann', 'maurer', 'steinmetz', 'tischler', 'schreiner',
        'schlosser', 'schmiede', 'werkstatt', 'handwerk', 'handwerker',
        'gewerbetreiber', 'weber', 'leineweber', 'schneider','architekt',
        'ziegel', 'sattler', 'hutmacher','gerbereien','holz', 'sägerei', 'kohle',
    ],
    'Handel': [
        'kaufmann', 'händler', 'krämer', 'verkäufer', 'kramerstoffe',
        'handel', 'markt', 'messe', 'mustermesse', 'waren', 'gewandhaus','münz',
    ],
    'Religion': [
        'pfarrer', 'priester', 'dompfarrer', 'kirche', 'dom', 'kapelle',
        'kloster', 'münster', 'jesuitenweg', 'johanneskirche', 'pauluskirche',
        'martinskirche', 'theodor', 'martin', 'johannes', 'jacob', 'jakob',
        'elisabet', 'felix', 'nikolaus', 'franziskanerplatz', 'johanniter',
        'Gott', 'evangel', 'reform','fasnacht', 'religion', 'religiöser', 
    ],
    'Politik': [
        'bürgermeister', 'zunftmeister', 'rat', 'rathaus', 'ratsherr',
        'zunftherr', 'bürger', 'meister', 'präsident', 'kanzler', 'politik',
    ],
    'Adel': [
        'könig', 'königin', 'kaiser', 'kaiserin', 'prinz', 'prinzessin',
        'graf', 'gräfin', 'freiherr', 'freifrau', 'baron', 'baronin',
        'herzog', 'herzogin', 'führer', 'fürstin', 'burgunder', 'habsburg', 'ritter','Marschalk',
    ],
    'Militär': [
        'general', 'oberst', 'major', 'hauptmann', 'leutnant', 'soldat',
        'kriegrat', 'feldherr', 'festung', 'krieg', 'fort', 'befestigung','schützen',
        'schiess','offizier', 'wache', 'kaserne', 'arsenal', 'zeughaus', 'batterie','kommandant',
    ],
    'Ortschaft': [
        'basel', 'bern', 'zürich', 'luzern', 'st.gallen', 'appenzell',
        'wallis', 'graubünden', 'tessin', 'jura', 'fribourg',
        'biel', 'solothurn', 'aarau', 'liestal', 'bellinzona',
        'deutsches', 'belgien', 'holland', 'frankreich', 'italien', 'gemeinde', 'dornach', 'stadt', 'Dorf',
    ],
    'Geografie': [ 'schwarzwald', 'vogesen', 'jura', 'alpen', 'berg', 'tal', 
                  
    ],
    'Gebäude': [
        'schloss', 'burg', 'festung', 'turm', 'brücke', 'rathaus',
        'kirche', 'dom', 'kapelle', 'kloster', 'münster', 'opernhaus',
        'theater', 'galerie', 'museum', 'bibliothek', 'archiv',
        'arsenalstrasse', 'arsenal', 'zeughaus', 'kaserne',
        'spital', 'spitalgasse', 'wasenplatz', 'waaghaus',
        'zunfthaus', 'gildenhaus', 'gilde', 'gewandhaus',
        'marktplatz', 'markthalle', 'stall', 'magazin',
        'mühle', 'darre', 'backhaus', 'brauerei', 'brennerei',
        'färberei', 'tuchhandel', 'tuchschau', 'walke', 'bahnhof', 'haus', 'hof','scheuene', 
    ],
    
    'Tiere': [
        'wolf', 'hirsch', 'fuchs', 'löwe',
        'eichhorn', 'wild', 'pferd', 'ross', 'hund',
        'vogel', 'adler', 'falke', 'ente',
        'fisch', 'forelle',
    ],
    'Pflanzen': [
        'eiche', 'eich', 'linde', 'birke', 'erle', 'erlen',
        'tanne', 'buche', 'kastanie', 'ahorn', 'ulme',
        'rose', 'weide', 'busch',
        'rebe', 'wein',
        'baum', 'wald', 'gras', 'pflanze', 'blume',
    ],
    'Gewässer': [
        # Flüsse in Basel
        'rhein', 'birs', 'birsig', 'wiese',

        'bach', 'fluss', 'teich', 'quelle', 'brunnen',
        'wasser', 'zufluss',
    ],
}

Geschlecht zuordnen

In [ ]:
import gender_guesser.detector as gender
import re

detector = gender.Detector(case_sensitive=False)

def geschlecht_erkennen(text):
    if not text or not str(text).strip():
        return 'unbekannt'
    erstes_wort = str(text).split()[0]      # Vorname steht meist am Anfang
    g = detector.get_gender(erstes_wort)
    if g in ('male', 'mostly_male'):
        return 'männlich'
    if g in ('female', 'mostly_female'):
        return 'weiblich'
    return 'unbekannt'

data["geschlecht"] = data["Erklärung_komplett"].apply(geschlecht_erkennen)
print(data["geschlecht"].value_counts().to_string())

Kategorien zuordnen

In [ ]:
def berufsgruppe_erkennen(text):
    """Gibt die erste passende Berufsgruppe zurück, sonst 'Sonstiges'."""
    if not text: return 'Sonstiges'
    t = text.lower()
    for gruppe, woerter in PROFESSIONS.items():
        if any(re.search(w, t) for w in woerter):
            return gruppe
    return 'Sonstiges'

data["berufsgruppe"] = data["Erklärung_komplett"].apply(berufsgruppe_erkennen)

print(data["berufsgruppe"].value_counts().to_string())

Jahreszahl in Epoche

In [ ]:
def year_to_epoch(year) -> str:
    """Wandle eine Jahreszahl in eine Epoche um."""
    if pd.isna(year):
        return None
    try:
        y = int(year)
    except (ValueError, TypeError):
        return None

    if y < 1500:
        return 'Mittelalter'
    elif y < 1600:
        return 'Renaissance'
    elif y < 1750:
        return 'Barock'
    elif y < 1830:
        return 'Klassizismus'
    elif y < 1900:
        return '19. Jahrhundert'
    elif y < 1945:
        return 'Moderne'
    else:
        return 'Zeitgenössisch'

Epoche zuordnen

In [ ]:
import pandas as pd

def classify_epoch(erstmals_erwaehnt=None, amtlich_benannt=None):
    """Bestimme Epoche anhand von Jahreszahlen."""
    if erstmals_erwaehnt:
        epoch = year_to_epoch(erstmals_erwaehnt) 
        if epoch:
            return epoch
    elif amtlich_benannt:
        epoch = year_to_epoch(amtlich_benannt)
        if epoch:
            return epoch
    else:
        return "keine Angabe"
    
data["epoche"] = data.apply(
    lambda row: classify_epoch(
        row["Erstmals erwähnt"],
        row["Amtlich benannt"]
    ),
    axis=1
)


In [ ]:
data.to_csv("data/CSV_fuer_Streamlint.csv", sep=";", index=False, encoding="utf-8-sig")

## FOLIUM Karte erstellen

In [ ]:
import folium
from folium import Element

m = folium.Map(location=(47.554554, 7.590270), zoom_start=14, tiles="CartoDB positron", attr="CartoDB positron")

for _, zeile in data.iterrows():

    folium.GeoJson(
        zeile["Geo Shape"],
        tooltip=zeile["Strassenname"],
        control=False,
    ).add_to(m)


m.get_root().html.add_child(Element("""
<div style="
    position: fixed;
    top: 80px;
    left: 20px;
    z-index:9999;
    font-size:30px;
">
↑<br>N
</div>
"""))

poi_layer = folium.FeatureGroup(name="PointsOfInterest", show=False).add_to(m)

folium.Marker(
    location=(47.552230, 7.587495),
    tooltip="Balz, Besucheranzahl: 67"
).add_to(poi_layer)

folium.Marker(
    location=(47.553476, 7.585683),
    tooltip="Valhalla"
).add_to(poi_layer)

folium.LayerControl(collapsed=False).add_to(m)

m


In [ ]:
m.save("data/streetlore_map.html")

In [ ]:
# 1. Karte erstellen
m = folium.Map(location=(47.554554, 7.590270), zoom_start=14, tiles="CartoDB positron", attr="CartoDB positron")

# 2. Farb-Mapping definieren (Trage hier deine Kategorien und Wunschfarben ein)
farb_zuordnung = {
    'männlich': '#3182bd',      # Blau
    'weiblich': '#e377c2',      # Rosa
    'unbekannt': '#969696',     # Grau
}

# 3. Schleife über das DataFrame
for _, zeile in data.iterrows():
    
    # Kategorie aus der aktuellen Zeile holen (hier "geschlecht", für Epochen einfach anpassen)
    kategorie = zeile["geschlecht"]
    
    # Farbe aus dem Mapping holen.
    linie_farbe = farb_zuordnung.get(kategorie)
    
    folium.GeoJson(
        zeile["Geo Shape"],
        tooltip=zeile["Strassenname"],
        # Hier wird das Styling für die Linie definiert:
        style_function=lambda x, farbe=linie_farbe: {
            'color': farbe,          # Farbe der Linie
            'weight': 4,             # Dicke der Linie (höher = dicker)
            'opacity': 0.8           # Deckkraft (0.0 bis 1.0)
        },
        control=False,
    ).add_to(m)

legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 200px; height: 100px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.85;">
     &nbsp;<b>Legende</b><br>
    <i class="fa fa-circle" style="color:#e377c2"></i>
    &nbsp;Frauen<br>
    <i class="fa fa-circle" style="color:#3182bd"></i>
    &nbsp;Männer<br>
    <i class="fa fa-circle" style="color:#969696"></i>
    &nbsp;Unbekannt
</div>
'''

# Add the legend to the map
m.get_root().html.add_child(folium.Element(legend_html))

m.get_root().html.add_child(Element("""
<div style="
    position: fixed;
    top: 80px;
    left: 20px;
    z-index:9999;
    font-size:30px;
">
↑<br>N
</div>
"""))

poi_layer = folium.FeatureGroup(name="PointsOfInterest").add_to(m)

folium.Marker(
    location=(47.552230, 7.587495),
    tooltip="Balz"
).add_to(poi_layer)

folium.Marker(
    location=(47.553476, 7.585683),
    tooltip="Valhalla"
).add_to(poi_layer)

folium.LayerControl(collapsed=False).add_to(m)

# Karte anzeigen
m

In [ ]:
m.save("data/streetlore_gender_map.html")

In [ ]:
from folium import FeatureGroup, LayerControl
# 1. Karte erstellen
m = folium.Map(location=(47.554554, 7.590270), zoom_start=13, tiles="CartoDB positron", attr="CartoDB positron")

# 2. Farb-Mapping definieren (Trage hier deine Kategorien und Wunschfarben ein)
farb_zuordnung = {
    'Wissenschaft': "#3B82F6",   
    'Kunst': "#F59E0B",          
    'Handwerk': "#B45309",       
    'Handel': "#8B5CF6",    
    'Religion': "#EC4899",  
    'Politik': "#DC2626",   
    'Adel': "#06B6D4",      
    'Militär': "#374151",
    'Ortschaft': "#D97706",  
    'Geografie': "#16A34A",      
    'Pflanzen': "#22C55E",       
    'Tiere': "#A16207",          
    'Gewässer': "#0EA5E9",       
    'Gebäude': "#78716C",          
    'Sonstiges': "#9CA3AF"       
}

# Für jede Kategorie eigene Layer-Gruppe
gruppen = {}

for kategorie in farb_zuordnung.keys():

    gruppe = FeatureGroup(name=kategorie)

    gruppen[kategorie] = gruppe

    gruppe.add_to(m)


# Linien hinzufügen
for _, zeile in data.iterrows():

    kategorie = zeile["berufsgruppe"]

    farbe = farb_zuordnung.get(kategorie, "#999999")

    folium.GeoJson(

        zeile["Geo Shape"],

        tooltip=zeile["Erklärung_komplett"],

        style_function=lambda x, f=farbe: {
            "color": f,
            "weight": 4,
            "opacity": 0.8
        }

    ).add_to(gruppen[kategorie])


# Layer-Umschalter
LayerControl(collapsed=False).add_to(m)


legend_html = '''
<div style="position: fixed;
    bottom: 50px;
    left: 50px;
    width: 220px;
    padding: 10px;
    border: 2px solid grey;
    border-radius: 8px;
    z-index: 9999;
    font-size: 13px;
    background-color: white;
    opacity: 0.9;
    line-height: 1.6;;">
     &nbsp;<b>Legende</b><br><br>
    <i class="fa fa-circle" style="color:#3B82F6"></i>
    &nbsp; Wissenschaft<br>
    <i class="fa fa-circle" style="color:#F59E0B"></i>
    &nbsp; Kunst<br>
    <i class="fa fa-circle" style="color:#B45309"></i>
    &nbsp; Handwerk<br>
    <i class="fa fa-circle" style="color:#8B5CF6"></i>
    &nbsp; Handel<br>
    <i class="fa fa-circle" style="color:#EC4899"></i>
    &nbsp; Religion<br>
    <i class="fa fa-circle" style="color:#DC2626"></i>
    &nbsp; Politik<br>
    <i class="fa fa-circle" style="color:#06B6D4"></i>
    &nbsp; Adel<br>
    <i class="fa fa-circle" style="color:#374151"></i>
    &nbsp; Militär<br>
    <i class="fa fa-circle" style="color:#16A34A"></i>
    &nbsp; Geografie<br>
    <i class="fa fa-circle" style="color:#D97706"></i>
    &nbsp; Ortschaft<br>
    <i class="fa fa-circle" style="color:#22C55E"></i>
    &nbsp; Pflanzen<br>
    <i class="fa fa-circle" style="color:#A16207"></i>
    &nbsp; Tiere<br>
    <i class="fa fa-circle" style="color:#0EA5E9"></i>
    &nbsp; Gewässer<br>
    <i class="fa fa-circle" style="color:#78716C"></i>
    &nbsp; Gebäude<br>
    <i class="fa fa-circle" style="color:#14B8A6"></i>
    &nbsp; Epoche<br>
    <i class="fa fa-circle" style="color:#9CA3AF"></i>
    &nbsp; Sonstiges
    </div>
    '''

# Add the legend to the map
m.get_root().html.add_child(folium.Element(legend_html))

m.get_root().html.add_child(Element("""
<div style="
    position: fixed;
    top: 80px;
    left: 20px;
    z-index:9999;
    font-size:30px;
">
↑<br>N
</div>
"""))

# Karte anzeigen
m

In [ ]:
m.save("data/streetlore_kategorien_map.html")

In [ ]:
# Karte
m = folium.Map(
    location=(47.554554, 7.590270),
    zoom_start=13,
    tiles="CartoDB positron"
)

# Farben für Epochen
farb_zuordnung = {
    "Antike": "#8B5CF6",
    "Mittelalter": "#B45309",
    "Renaissance": "#F59E0B",
    "Barock": "#EC4899",
    "Klassizismus": "#3B82F6",
    "19. Jahrhundert": "#16A34A",
    "Moderne": "#DC2626",
    "keine Angabe": "#9CA3AF"
}

# Linien zeichnen
for _, zeile in data.iterrows():

    epoche = zeile["epoche"]

    linie_farbe = farb_zuordnung.get(
        epoche,
        "#999999"
    )
    folium.GeoJson(

        zeile["Geo Shape"],

        tooltip=(
            f"{zeile['Strassenname']}<br>"
            f"{epoche}"
        ),
        style_function=lambda x, farbe=linie_farbe: {
            "color": farbe,
            "weight": 4,
            "opacity": 0.8
        },

    ).add_to(m)


# Legende
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 220px;
    padding: 10px;
    border: 2px solid grey;
    border-radius: 8px;
    z-index: 9999;
    font-size: 13px;
    background-color: white;
    opacity: 0.9;
    line-height: 1.6;
">
<b>Legende – Epochen</b><br><br>
<i class="fa fa-circle" style="color:#8B5CF6"></i>
&nbsp; Antike<br>
<i class="fa fa-circle" style="color:#B45309"></i>
&nbsp; Mittelalter<br>
<i class="fa fa-circle" style="color:#F59E0B"></i>
&nbsp; Renaissance<br>
<i class="fa fa-circle" style="color:#EC4899"></i>
&nbsp; Barock<br>
<i class="fa fa-circle" style="color:#3B82F6"></i>
&nbsp; Klassizismus<br>
<i class="fa fa-circle" style="color:#16A34A"></i>
&nbsp; 19. Jahrhundert<br>
<i class="fa fa-circle" style="color:#DC2626"></i>
&nbsp; Moderne<br>
<i class="fa fa-circle" style="color:#9CA3AF"></i>
&nbsp; keine Angabe
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# Nordpfeil
m.get_root().html.add_child(Element("""
<div style="
    position: fixed;
    top: 80px;
    left: 20px;
    z-index:9999;
    font-size:30px;
">
↑<br>N
</div>
"""))

m